# South Kensington Rollout-finetuned Surrogate Test

This notebook compares the rollout-finetuned temperature surrogate against the South Kensington physical 3D temperature result. It uses an autoregressive rollout: only the first `HISTORY_STEPS` physical frames initialize the surrogate, then surrogate predictions are fed back into later steps.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import Image as IPyImage, display

cwd = Path.cwd().resolve()
if (cwd / 'test_virtual_geometry_rollout_surrogate.py').exists():
    ROLLOUT_DIR = cwd
elif (cwd / 'rollout_surrogate' / 'test_virtual_geometry_rollout_surrogate.py').exists():
    ROLLOUT_DIR = cwd / 'rollout_surrogate'
else:
    raise FileNotFoundError('Open this notebook from temperature_field_bundle or rollout_surrogate.')

if str(ROLLOUT_DIR) not in sys.path:
    sys.path.insert(0, str(ROLLOUT_DIR))

import test_virtual_geometry_rollout_surrogate as t

SK_CASE_DIR = t.BUNDLE_ROOT / 'outputs' / 'south_kensington_temperature_3d'
OUT_DIR = t.ROLLOUT_RUN_DIR / 'south_kensington_rollout_test'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Rollout dir:', ROLLOUT_DIR)
print('Bundle root:', t.BUNDLE_ROOT)
print('South Kensington case:', SK_CASE_DIR)
print('Output dir:', OUT_DIR)
print('Rollout checkpoint:', t.ROLLOUT_CHECKPOINT)

## Load Checkpoint and South Kensington Case

In [ ]:
required = [
    SK_CASE_DIR / 'temperature_fields_3d_c.npy',
    SK_CASE_DIR / 'temperature_3d_run_summary.json',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Missing South Kensington physical outputs. Run Run_South_Kensington_Temperature_3D_Labels.ipynb first. Missing:\n'
        + '\n'.join(missing)
    )

checkpoint_path = t.ROLLOUT_CHECKPOINT if t.ROLLOUT_CHECKPOINT.exists() else t.FALLBACK_CHECKPOINT
if checkpoint_path == t.FALLBACK_CHECKPOINT:
    print('Rollout checkpoint not found; falling back to one-step checkpoint for comparison.')

checkpoint = torch.load(checkpoint_path, map_location='cpu')
stats = t.SurrogateStats(**checkpoint['stats'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = t.load_model_from_checkpoint(checkpoint, device)

case = t.apply_z_crop(t.Temperature3DCase(SK_CASE_DIR))
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('Loaded checkpoint:', checkpoint_path)
print('Checkpoint epoch:', checkpoint.get('epoch', 'unknown'))
print('Z crop:', t.Z_CROP)
print('Temperature shape:', case.temperature.shape)
print('Velocity shape:', case.u.shape)

## Autoregressive Rollout

In [ ]:
pred_seq, true_seq, valid_mask, frame_ids = t.autoregressive_rollout(case, model, stats, device)
err_seq = pred_seq - true_seq

metrics = {
    'checkpoint': str(checkpoint_path),
    'case_dir': str(SK_CASE_DIR),
    'mode': 'south_kensington_autoregressive_rollout',
    'use_next_velocity': t.USE_NEXT_VELOCITY,
    'correction_iterations': t.CORRECTION_ITERATIONS,
    'z_crop': t.Z_CROP,
    'patch_size': list(t.PATCH_SIZE),
    'use_cool_surface_channels': t.USE_COOL_SURFACE_CHANNELS,
    'use_cool_surface_relaxation': t.USE_COOL_SURFACE_RELAXATION,
    'cool_surface_relaxation': t.COOL_SURFACE_RELAXATION,
    'open_ground_relaxation': t.OPEN_GROUND_RELAXATION,
    'cool_surface_vertical_decay_layers': t.COOL_SURFACE_VERTICAL_DECAY_LAYERS,
    'in_channels': t.input_channel_count(t.HISTORY_STEPS),
    'frames': [int(v) for v in frame_ids],
    'first_predicted_frame': int(frame_ids[0]),
    'first_step_bias_c': float(np.mean(err_seq[0][valid_mask])),
    'first_step_mae_c': float(np.mean(np.abs(err_seq[0][valid_mask]))),
    'first_step_rmse_c': float(np.sqrt(np.mean(err_seq[0][valid_mask] ** 2))),
    'final_frame': int(frame_ids[-1]),
    'final_bias_c': float(np.mean(err_seq[-1][valid_mask])),
    'final_mae_c': float(np.mean(np.abs(err_seq[-1][valid_mask]))),
    'final_rmse_c': float(np.sqrt(np.mean(err_seq[-1][valid_mask] ** 2))),
    'rollout_bias_c': float(np.mean(err_seq[:, valid_mask])),
    'rollout_mae_c': float(np.mean(np.abs(err_seq[:, valid_mask]))),
    'rollout_rmse_c': float(np.sqrt(np.mean(err_seq[:, valid_mask] ** 2))),
}
metrics_path = OUT_DIR / 'south_kensington_rollout_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))
print('Saved metrics:', metrics_path)

## Final-frame Horizontal Panels

In [ ]:
velocity_dir = case._velocity_cache_dir()
building_mask_path = velocity_dir / 'building_mask_2d.npy'
building_mask_2d = np.load(building_mask_path).astype(bool) if building_mask_path.exists() else np.any(case.solid_mask, axis=0)

# Temporarily redirect the reusable plotting helper to this notebook's output directory.
old_out_dir = t.OUT_DIR
t.OUT_DIR = OUT_DIR
t.save_final_panel(pred_seq[-1], true_seq[-1], valid_mask, int(frame_ids[-1]), building_mask_2d)
panel_path = OUT_DIR / 'final_frame_horizontal_panels.png'
print('Saved panel:', panel_path)
display(IPyImage(filename=str(panel_path)))
t.OUT_DIR = old_out_dir

## Error Through Time

In [ ]:
frame_mae = np.asarray([np.mean(np.abs(err_seq[i][valid_mask])) for i in range(err_seq.shape[0])], dtype=np.float32)
frame_bias = np.asarray([np.mean(err_seq[i][valid_mask]) for i in range(err_seq.shape[0])], dtype=np.float32)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(frame_ids, frame_mae, marker='o')
axes[0].set_title('South Kensington rollout MAE')
axes[0].set_xlabel('frame')
axes[0].set_ylabel('deg C')
axes[0].grid(True, alpha=0.3)
axes[1].plot(frame_ids, frame_bias, marker='o')
axes[1].axhline(0.0, color='black', linewidth=0.8)
axes[1].set_title('South Kensington rollout bias')
axes[1].set_xlabel('frame')
axes[1].set_ylabel('deg C')
axes[1].grid(True, alpha=0.3)
plt.show()